In [1]:
import pandas as pd 
import numpy as np 
import json  


In [2]:
cd "C:\Users\Trisha Tyagi\OneDrive\Desktop\QueryClarify"

C:\Users\Trisha Tyagi\OneDrive\Desktop\QueryClarify


In [5]:

with open("spider_data\dev.json", "r") as f:
    dev_data = json.load(f)

with open(r"spider_data\tables.json", "r") as f:
    tables_data = json.load(f)

with open(r"spider_data\train_spider.json", "r") as f:
    train_data = json.load(f)

In [6]:
print(len(dev_data))
print(len(tables_data))
print(len(train_data))

print(dev_data[0].keys())
print(tables_data[0].keys())
print(train_data[0].keys())

1034
166
7000
dict_keys(['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql'])
dict_keys(['column_names', 'column_names_original', 'column_types', 'db_id', 'foreign_keys', 'primary_keys', 'table_names', 'table_names_original'])
dict_keys(['db_id', 'query', 'query_toks', 'query_toks_no_value', 'question', 'question_toks', 'sql'])


In [10]:
print(type(dev_data))

<class 'list'>


In [9]:
schema_mapping = {}

for schema in tables_data:
    db_id = schema["db_id"]
    schema_mapping[db_id] = schema

In [10]:
print("Total databases:", len(schema_mapping))

first_db_id = list(schema_mapping.keys())[0]

print("Database ID:", first_db_id)
print("Corresponding schema:", schema_mapping[first_db_id])

Total databases: 166
Database ID: perpetrator
Corresponding schema: {'column_names': [[-1, '*'], [0, 'perpetrator id'], [0, 'people id'], [0, 'date'], [0, 'year'], [0, 'location'], [0, 'country'], [0, 'killed'], [0, 'injured'], [1, 'people id'], [1, 'name'], [1, 'height'], [1, 'weight'], [1, 'home town']], 'column_names_original': [[-1, '*'], [0, 'Perpetrator_ID'], [0, 'People_ID'], [0, 'Date'], [0, 'Year'], [0, 'Location'], [0, 'Country'], [0, 'Killed'], [0, 'Injured'], [1, 'People_ID'], [1, 'Name'], [1, 'Height'], [1, 'Weight'], [1, 'Home Town']], 'column_types': ['text', 'number', 'number', 'text', 'number', 'text', 'text', 'number', 'number', 'number', 'text', 'number', 'number', 'text'], 'db_id': 'perpetrator', 'foreign_keys': [[2, 9]], 'primary_keys': [1, 9], 'table_names': ['perpetrator', 'people'], 'table_names_original': ['perpetrator', 'people']}


In [ ]:
# for data in dev_data:
#     db=data['db_id']
#     schema=schema_mapping[db]
#     question=data['question']
#     query=data['query']

#     print(db)
#     print(question)
#     print(query)
#     print(schema)

In [16]:
def format_schema(schema):
    schema_text = "Database Schema\n"

    for i, table in enumerate(schema['table_names_original']):
        schema_text += f"\nTable: {table}\n"
        schema_text += "Columns:\n"

        for col_idx, (table_idx, column) in enumerate(schema['column_names_original']):
            if table_idx == i:
                dtype = schema['column_types'][col_idx]
                schema_text += f"- {column} ({dtype})\n"

    schema_text += "\nForeign Keys:\n"
    for f_key in schema['foreign_keys']:
        table_idx1, col1 = schema['column_names_original'][f_key[0]]
        table_idx2, col2 = schema['column_names_original'][f_key[1]]

        table1 = schema['table_names_original'][table_idx1]
        table2 = schema['table_names_original'][table_idx2]

        schema_text += f"- {table1}.{col1} -> {table2}.{col2}\n"

    schema_text += "\nPrimary Keys:\n"
    for p_key in schema['primary_keys']:
        table_idx, column = schema['column_names_original'][p_key]
        table = schema['table_names_original'][table_idx]

        schema_text += f"- {table}.{column}\n"

    return schema_text

In [22]:
processed_data_dev = []

for data in dev_data:
    db = data['db_id']
    schema = schema_mapping[db]
    question = data['question']
    query = data['query']

    database_schema = format_schema(schema)

    example = {
        "instruction": "Generate SQL for the given question using the database schema.",
        "input": f"{database_schema}\nQuestion: {question}",
        "output": query
    }

    processed_data_dev.append(example)

In [17]:
processed_data = []

for data in train_data:
    db = data['db_id']
    schema = schema_mapping[db]
    question = data['question']
    query = data['query']

    database_schema = format_schema(schema)

    example = {
        "instruction": "Generate SQL for the given question using the database schema.",
        "input": f"{database_schema}\nQuestion: {question}",
        "output": query
    }

    processed_data.append(example)

In [12]:
processed_data

[{'instruction': 'Generate SQL for the given question using the database schema.',
  'input': 'Database Schema\n\nTable: department\nColumns:\n- department id (number)\n- name (text)\n- creation (text)\n- ranking (number)\n- budget in billions (number)\n- num employees (number)\n\nTable: head\nColumns:\n- head id (number)\n- name (text)\n- born state (text)\n- age (number)\n\nTable: management\nColumns:\n- department id (number)\n- head id (number)\n- temporary acting (text)\n\nForeign Keys:\n- head id -> head id\n- department id -> department id\n\nPrimary Keys:\n- department id\n- head id\n- department id\n\nQuestion: How many heads of the departments are older than 56 ?',
  'output': 'SELECT count(*) FROM head WHERE age  >  56'},
 {'instruction': 'Generate SQL for the given question using the database schema.',
  'input': 'Database Schema\n\nTable: department\nColumns:\n- department id (number)\n- name (text)\n- creation (text)\n- ranking (number)\n- budget in billions (number)\n- n

In [29]:
with open("processed_dev.json", "w") as f:
    json.dump(processed_data_dev, f, indent=4)

In [19]:
with open("processed_train.json", "w") as f:
    json.dump(processed_data, f, indent=4)

In [25]:
import os

print("File exists:", os.path.exists("processed_dev.json"))
print("Total examples:", len(processed_data))

File exists: True
Total examples: 7000


In [26]:
for example in processed_data[:3]:
    print(example)
    print("-" * 50)

{'instruction': 'Generate SQL for the given question using the database schema.', 'input': 'Database Schema\n\nTable: department\nColumns:\n- Department_ID (number)\n- Name (text)\n- Creation (text)\n- Ranking (number)\n- Budget_in_Billions (number)\n- Num_Employees (number)\n\nTable: head\nColumns:\n- head_ID (number)\n- name (text)\n- born_state (text)\n- age (number)\n\nTable: management\nColumns:\n- department_ID (number)\n- head_ID (number)\n- temporary_acting (text)\n\nForeign Keys:\n- management.head_ID -> head.head_ID\n- management.department_ID -> department.Department_ID\n\nPrimary Keys:\n- department.Department_ID\n- head.head_ID\n- management.department_ID\n\nQuestion: How many heads of the departments are older than 56 ?', 'output': 'SELECT count(*) FROM head WHERE age  >  56'}
--------------------------------------------------
{'instruction': 'Generate SQL for the given question using the database schema.', 'input': 'Database Schema\n\nTable: department\nColumns:\n- Depar

In [27]:


with open("processed_train.json", "r") as f:
    processed_train = json.load(f)

print("Total examples:", len(processed_train))

print("Missing instruction:",
      sum(not x.get("instruction") for x in processed_train))

print("Missing input:",
      sum(not x.get("input") for x in processed_train))

print("Missing output:",
      sum(not x.get("output") for x in processed_train))

Total examples: 7000
Missing instruction: 0
Missing input: 0
Missing output: 0


In [30]:
for name, data in [
    ("Train", processed_data),
    ("Dev", processed_data_dev)
]:
    unique = len(set(
        (x["input"], x["output"]) for x in data
    ))

    print(name)
    print("Total:", len(data))
    print("Unique:", unique)
    print("Duplicates:", len(data) - unique)

Train
Total: 7000
Unique: 6992
Duplicates: 8
Dev
Total: 1034
Unique: 1034
Duplicates: 0


In [31]:
invalid_db = []

for item in train_data:
    if item["db_id"] not in schema_mapping:
        invalid_db.append(item["db_id"])

print("Invalid database references:", len(invalid_db))

Invalid database references: 0


In [32]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print(tokenizer.chat_template)

Tokenizer loaded successfully!
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou

In [33]:
system_prompt = """You are a Text-to-SQL assistant.
Your task is to generate a correct SQL query based on the given database schema and natural language question.

Instructions:
1. Use only tables and columns present in the provided schema.
2. Follow the relationships defined by foreign keys when joining tables.
3. Generate SQL that answers the user's question accurately.
4. Do not invent tables, columns, or values.
5. Return only the SQL query without explanations or markdown formatting.
"""

In [34]:
formatted_train = []

for item in processed_train:
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": item["input"]
        },
        {
            "role": "assistant",
            "content": item["output"]
        }
    ]

    formatted_train.append({"messages": messages})

print("Total formatted examples:", len(formatted_train))
print(formatted_train[0])

Total formatted examples: 7000
{'messages': [{'role': 'system', 'content': "You are a Text-to-SQL assistant.\nYour task is to generate a correct SQL query based on the given database schema and natural language question.\n\nInstructions:\n1. Use only tables and columns present in the provided schema.\n2. Follow the relationships defined by foreign keys when joining tables.\n3. Generate SQL that answers the user's question accurately.\n4. Do not invent tables, columns, or values.\n5. Return only the SQL query without explanations or markdown formatting.\n"}, {'role': 'user', 'content': 'Database Schema\n\nTable: department\nColumns:\n- Department_ID (number)\n- Name (text)\n- Creation (text)\n- Ranking (number)\n- Budget_in_Billions (number)\n- Num_Employees (number)\n\nTable: head\nColumns:\n- head_ID (number)\n- name (text)\n- born_state (text)\n- age (number)\n\nTable: management\nColumns:\n- department_ID (number)\n- head_ID (number)\n- temporary_acting (text)\n\nForeign Keys:\n- ma

In [35]:
formatted_dev = []

for item in processed_data_dev:
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": item["input"]
        },
        {
            "role": "assistant",
            "content": item["output"]
        }
    ]

    formatted_dev.append({"messages": messages})

print("Total formatted examples:", len(formatted_dev))
print(formatted_dev[0])

Total formatted examples: 1034
{'messages': [{'role': 'system', 'content': "You are a Text-to-SQL assistant.\nYour task is to generate a correct SQL query based on the given database schema and natural language question.\n\nInstructions:\n1. Use only tables and columns present in the provided schema.\n2. Follow the relationships defined by foreign keys when joining tables.\n3. Generate SQL that answers the user's question accurately.\n4. Do not invent tables, columns, or values.\n5. Return only the SQL query without explanations or markdown formatting.\n"}, {'role': 'user', 'content': 'Database Schema\n\nTable: stadium\nColumns:\n- Stadium_ID (number)\n- Location (text)\n- Name (text)\n- Capacity (number)\n- Highest (number)\n- Lowest (number)\n- Average (number)\n\nTable: singer\nColumns:\n- Singer_ID (number)\n- Name (text)\n- Country (text)\n- Song_Name (text)\n- Song_release_year (text)\n- Age (number)\n- Is_male (others)\n\nTable: concert\nColumns:\n- concert_ID (number)\n- concer

In [36]:
formatted_text = tokenizer.apply_chat_template(
    formatted_train[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are a Text-to-SQL assistant.
Your task is to generate a correct SQL query based on the given database schema and natural language question.

Instructions:
1. Use only tables and columns present in the provided schema.
2. Follow the relationships defined by foreign keys when joining tables.
3. Generate SQL that answers the user's question accurately.
4. Do not invent tables, columns, or values.
5. Return only the SQL query without explanations or markdown formatting.
<|im_end|>
<|im_start|>user
Database Schema

Table: department
Columns:
- Department_ID (number)
- Name (text)
- Creation (text)
- Ranking (number)
- Budget_in_Billions (number)
- Num_Employees (number)

Table: head
Columns:
- head_ID (number)
- name (text)
- born_state (text)
- age (number)

Table: management
Columns:
- department_ID (number)
- head_ID (number)
- temporary_acting (text)

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> department.Department_ID

Primary

In [37]:
for item in formatted_train:
    item["text"] = tokenizer.apply_chat_template(
        item["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

for item in formatted_dev:
    item["text"] = tokenizer.apply_chat_template(
        item["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

print("Train:", len(formatted_train))
print("Dev:", len(formatted_dev))

print("\nSample:")
print(formatted_train[0]["text"])

Train: 7000
Dev: 1034

Sample:
<|im_start|>system
You are a Text-to-SQL assistant.
Your task is to generate a correct SQL query based on the given database schema and natural language question.

Instructions:
1. Use only tables and columns present in the provided schema.
2. Follow the relationships defined by foreign keys when joining tables.
3. Generate SQL that answers the user's question accurately.
4. Do not invent tables, columns, or values.
5. Return only the SQL query without explanations or markdown formatting.
<|im_end|>
<|im_start|>user
Database Schema

Table: department
Columns:
- Department_ID (number)
- Name (text)
- Creation (text)
- Ranking (number)
- Budget_in_Billions (number)
- Num_Employees (number)

Table: head
Columns:
- head_ID (number)
- name (text)
- born_state (text)
- age (number)

Table: management
Columns:
- department_ID (number)
- head_ID (number)
- temporary_acting (text)

Foreign Keys:
- management.head_ID -> head.head_ID
- management.department_ID -> de

In [38]:
with open("qwen_train.json", "w") as f:
    json.dump(formatted_train, f, indent=2)

with open("qwen_dev.json", "w") as f:
    json.dump(formatted_dev, f, indent=2)

print("Datasets saved successfully!")

Datasets saved successfully!


In [39]:
import json

with open("qwen_train.json", "r") as f:
    qwen_train = json.load(f)

with open("qwen_dev.json", "r") as f:
    qwen_dev = json.load(f)

for name, data in [
    ("Train", qwen_train),
    ("Dev", qwen_dev)
]:
    invalid = [
        i for i, item in enumerate(data)
        if "messages" not in item or "text" not in item
        or not item["text"].strip()
    ]

    print(name)
    print("Total:", len(data))
    print("Invalid examples:", len(invalid))

Train
Total: 7000
Invalid examples: 0
Dev
Total: 1034
Invalid examples: 0


In [40]:
def validate_dataset(name, data):
    invalid = []

    expected_roles = ["system", "user", "assistant"]

    for i, item in enumerate(data):

        if not all(key in item for key in ["messages", "text"]):
            invalid.append(i)
            continue

        messages = item["messages"]

        if not isinstance(messages, list) or len(messages) != 3:
            invalid.append(i)
            continue

        roles = [msg.get("role") for msg in messages]

        if roles != expected_roles:
            invalid.append(i)
            continue

        if any(
            not isinstance(msg.get("content"), str)
            or not msg["content"].strip()
            for msg in messages
        ):
            invalid.append(i)
            continue

        if not isinstance(item["text"], str) or not item["text"].strip():
            invalid.append(i)

    print(name)
    print("Total examples:", len(data))
    print("Invalid examples:", len(invalid))


validate_dataset("Train", qwen_train)
validate_dataset("Dev", qwen_dev)

Train
Total examples: 7000
Invalid examples: 0
Dev
Total examples: 1034
Invalid examples: 0


In [41]:
print("Training examples:", len(qwen_train))
print("Validation examples:", len(qwen_dev))

print("\nTraining count correct:",
      len(qwen_train) == len(processed_data))

print("Validation count correct:",
      len(qwen_dev) == len(processed_data_dev))

print("\nTraining mapping correct:",
      all(
          qwen_train[i]["messages"][1]["content"] == processed_data[i]["input"]
          and qwen_train[i]["messages"][2]["content"] == processed_data[i]["output"]
          for i in range(len(qwen_train))
      ))

print("Validation mapping correct:",
      all(
          qwen_dev[i]["messages"][1]["content"] == processed_data_dev[i]["input"]
          and qwen_dev[i]["messages"][2]["content"] == processed_data_dev[i]["output"]
          for i in range(len(qwen_dev))
      ))

Training examples: 7000
Validation examples: 1034

Training count correct: True
Validation count correct: True

Training mapping correct: True
Validation mapping correct: True
